In [1]:
%pip uninstall langchain langchain-community langchain-core -y

%pip install \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-core==0.3.66

Found existing installation: langchain 1.3.9
Uninstalling langchain-1.3.9:
  Successfully uninstalled langchain-1.3.9
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.7
Uninstalling langchain-core-1.4.7:
  Successfully uninstalled langchain-core-1.4.7
Note: you may need to restart the kernel to use updated packages.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 29.0 MB/s  0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.25.0
    Uninstalling zstandard-0.25.0:
      Successfully uninstalle

In [2]:
%pip install ragas

  Using cached jiter-0.14.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.2 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.3.0-py3-none-any.whl.metadata (3.1 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 53.5 MB/s  0:00:00m0:00:0100:01
Using cached jiter-0.14.0-cp311-cp311-macosx_11_0_ar

In [3]:
reference_data = [
  {
    "question": "What’s the leave policy?", #Office condos for sale in 94539
    "ground_truth": "employees must submit a leave request for approval.", #Expected llm context
    #"context": "Employees must submit a leave request for approval. " #Expected context from source documents based on the question #2 properties
  }
]
question = reference_data[0]['question']
ground_truth = reference_data[0]['ground_truth']
context = ground_truth #reference_data[0]['context']
print (f"question: {question}")
print (f"ground_truth: {ground_truth}")
print (f"context: {context}")

question: What’s the leave policy?
ground_truth: employees must submit a leave request for approval.
context: employees must submit a leave request for approval.


In [ ]:
# Retrieve context from Milvus DB

from milvus_chatbot_with_rag import retrieve_similiar_contexts

def perform_retrieval(question):

    retrieved_context = retrieve_similiar_contexts(question, "policy_docs_collection", 1)[0]['content']
    print (f"perform_retrieval.retrieved_context: {retrieved_context}")
    return retrieved_context

    
# retrieved_context = perform_retrieval("What’s the leave policy?")
retrieved_context = perform_retrieval(question)

ModuleNotFoundError: No module named 'milvus_chatbot_with_rag'

In [ ]:
# Validate retrieved context against reference data    

print(f"Retrieved context: {retrieved_context}")
print(f"Expected context: {context}")
print(f"question: {question}")

In [ ]:
%pip install datasets 

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

# question = reference_data[0]['question']
# ground_truth = reference_data[0]['ground_truth']
# context = reference_data[0]['context']
# Question User asked
question = reference_data[0]['question']

# Reference context (should be a string)
reference_context = reference_data[0]['ground_truth']

# Retrieved context (a string from perform_retrieval)
retrieved_context = [perform_retrieval(question)]

# Build dataset properly
dataset_dict = {
    "question": [question],
    "contexts": [retrieved_context],      # list of strings INSIDE another list
    "ground_truth": [reference_context],   # single string
    "answer": [""]
}

print(f"dataset_dict: {dataset_dict}")

ragas_dataset = Dataset.from_dict(dataset_dict)


# Evaluate retrieval
results = evaluate(
    dataset=ragas_dataset,
    metrics=[context_precision, context_recall]
)

In [ ]:
print("Retrieval Evaluation Results:")
results.to_pandas()